<a href="https://colab.research.google.com/github/eeeewyz/Audio-course/blob/main/ASR/whisper_finetuning_%E4%B8%8A%E6%B5%B7%E8%AF%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#加载一个上海话训练集
from datasets import load_dataset
dialect_corpus = load_dataset("TingChen-ppmc/Shanghai_Dialect_Conversational_Speech_Corpus")


README.md:   0%|          | 0.00/2.11k [00:00<?, ?B/s]

data/train-00000-of-00001-3b3c64c2b15eef(…): reconstructing file:   0%|          |  0.00B /  437MB            

data/train-00000-of-00001-3b3c64c2b15eef(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3792 [00:00<?, ? examples/s]

In [2]:
from datasets import load_dataset
train_split = load_dataset("TingChen-ppmc/Shanghai_Dialect_Conversational_Speech_Corpus", split="train")
# where test_size=0.3 denotes 0.7 of the dataset will be split to test split
corpus = train_split.train_test_split(test_size=0.3)


In [3]:
#我们只选择这两个metedata info
corpus = corpus.select_columns(["audio", "transcription"])

In [4]:
# corpus里面有训练集和测试集
print(corpus)

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 2654
    })
    test: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 1138
    })
})


In [5]:
from transformers.models.whisper.tokenization_whisper import TO_LANGUAGE_CODE

TO_LANGUAGE_CODE

{'english': 'en',
 'chinese': 'zh',
 'german': 'de',
 'spanish': 'es',
 'russian': 'ru',
 'korean': 'ko',
 'french': 'fr',
 'japanese': 'ja',
 'portuguese': 'pt',
 'turkish': 'tr',
 'polish': 'pl',
 'catalan': 'ca',
 'dutch': 'nl',
 'arabic': 'ar',
 'swedish': 'sv',
 'italian': 'it',
 'indonesian': 'id',
 'hindi': 'hi',
 'finnish': 'fi',
 'vietnamese': 'vi',
 'hebrew': 'he',
 'ukrainian': 'uk',
 'greek': 'el',
 'malay': 'ms',
 'czech': 'cs',
 'romanian': 'ro',
 'danish': 'da',
 'hungarian': 'hu',
 'tamil': 'ta',
 'norwegian': 'no',
 'thai': 'th',
 'urdu': 'ur',
 'croatian': 'hr',
 'bulgarian': 'bg',
 'lithuanian': 'lt',
 'latin': 'la',
 'maori': 'mi',
 'malayalam': 'ml',
 'welsh': 'cy',
 'slovak': 'sk',
 'telugu': 'te',
 'persian': 'fa',
 'latvian': 'lv',
 'bengali': 'bn',
 'serbian': 'sr',
 'azerbaijani': 'az',
 'slovenian': 'sl',
 'kannada': 'kn',
 'estonian': 'et',
 'macedonian': 'mk',
 'breton': 'br',
 'basque': 'eu',
 'icelandic': 'is',
 'armenian': 'hy',
 'nepali': 'ne',
 'mongol

In [6]:
#加载processor,包含了fearture extractor和tokenizer
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small", language="chinese", task="transcribe"
)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [7]:
#第一步：resampling
from datasets import Audio
#从whisper读取16khzsampling_rate
sampling_rate = processor.feature_extractor.sampling_rate

# 将数据集的 audio 列设置为读取时自动重采样到 16 kHz。
# 并不是马上把所有音频都重采样一遍，它只是给 audio 列设置一个新的读取规则：
# 以后读取 audio→ 自动以 16 kHz 返回
corpus = corpus.cast_column("audio", Audio(sampling_rate=sampling_rate))

In [8]:
#看下数据的基本情况
corpus['train'],corpus['test']

(Dataset({
     features: ['audio', 'transcription'],
     num_rows: 2654
 }),
 Dataset({
     features: ['audio', 'transcription'],
     num_rows: 1138
 }))

In [9]:
corpus['train']['audio']

Column([<datasets.features._torchcodec.AudioDecoder object at 0x7b27c6445550>, <datasets.features._torchcodec.AudioDecoder object at 0x7b27c693a710>, <datasets.features._torchcodec.AudioDecoder object at 0x7b27c693ad50>, <datasets.features._torchcodec.AudioDecoder object at 0x7b27c4fe88a0>, <datasets.features._torchcodec.AudioDecoder object at 0x7b27c4fe8d60>, ...])

In [10]:
sample1=corpus['train'][0]
sample1

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x7b29488b2450>,
 'transcription': '每天夜到就来困觉了，早哴头就跑出去了，人家门一开伊就跟出去了，现在呐搿猫呐已经养了老大了'}

In [11]:
#采样率
sample1["audio"]["sampling_rate"]

16000

In [12]:
# 例子
# len(audio["array"]) = 48000
# audio["sampling_rate"] = 16000
# input_length = 48000/16000= 3

In [13]:
# #

#     example = processor(
#         audio=audio["array"],
#         sampling_rate=audio["sampling_rate"],
#         text=example["sentence"],
#     )

# 这几行最核心的意思是：把“原始音频 + 对应文字”一起交给 processor，转换成 Whisper 训练时真正需要的模型输入和标签。

# 你可以直接这样备注：

# # 用 processor 同时处理音频和对应的转录文本，
# # 把原始数据转换成模型训练需要的格式：
# #   audio -> 提取音频特征（input_features）
# #   text  -> tokenizer 编码成 token IDs（labels）
# example = processor(
#     audio=audio["array"],                   # 原始音频波形
#     sampling_rate=audio["sampling_rate"],   # 音频采样率，例如 16000 Hz
#     text=example["transcription"],          # 这段音频对应的正确文本（ground truth）
# )

# 可以把它理解成下面这个流程：

# 原始样本
# │
# ├── audio["array"]
# │      ↓
# │   Feature Extractor
# │      ↓
# │   input_features     ← 给 Whisper Encoder
# │
# └── transcription
#        ↓
#     Tokenizer
#        ↓
#     labels             ← 给 Whisper Decoder 学习

# 比如原始数据是：

# {
#     "audio": {
#         "array": [...],
#         "sampling_rate": 16000
#     },
#     "transcription": "今天天气很好"
# }

# 执行：

# example = processor(...)

# 之后，example 就不再只是原来的 "audio" 和 "transcription" 结构了，而会变成适合 Whisper 训练的处理结果，核心就是：

# {
#     "input_features": ...,   # 音频特征
#     "labels": ...            # 文本对应的 token IDs
# }

# 所以你之前把它理解成“增加几个字段”还不完全准确。这里真正做的是数据预处理/格式转换：把人能理解的音频和文字，转换成模型能直接吃进去的数字。

In [14]:
#第二步：转化成whisper需要的格式
def prepare_dataset(example):
    # 取出原始音频信息：
    # audio["array"] = waveform
    # audio["sampling_rate"] = 采样率
    audio = example["audio"]

    # 将原始数据转换成 Whisper 训练需要的格式：
    # audio → input_features
    # transcription → labels
    example = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["transcription"],
    )

    # 额外记录音频时长（秒）
    example["input_length"] = len(audio["array"]) / audio["sampling_rate"]

    return example

In [15]:
corpus = corpus.map(
    prepare_dataset,
    # 对数据集中的每一条样本执行 prepare_dataset()
    # 原始 audio + transcription
    # → input_features + labels + input_length

    # 处理完成后，删除原来的字段，例如 audio、transcription 等，就是把原来 train 里的所有旧列名拿出来，等 prepare_dataset() 生成新字段后，把这些旧列删掉。
    remove_columns=corpus.column_names["train"],

    # 使用 1 个进程处理数据
    num_proc=1
)

Map (num_proc=1):   0%|          | 0/2654 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/1138 [00:00<?, ? examples/s]

In [16]:
#查看目前数据集的字段
corpus.column_names

{'train': ['input_features', 'labels', 'input_length'],
 'test': ['input_features', 'labels', 'input_length']}

In [17]:
# input_features 和 labels 都是这句 processor(...) 自动生成的：

# example = processor(
#     audio=audio["array"],
#     sampling_rate=audio["sampling_rate"],
#     text=example["transcription"],
# )

# 具体对应关系是：

# audio + sampling_rate
#         ↓
# processor.feature_extractor
#         ↓
# input_features

# 而：

# transcription
#         ↓
# processor.tokenizer
#         ↓
# labels

# 所以你虽然没有手动写：

# example["labels"] = ...

# 但因为你传了：

# text=example["transcription"]

# WhisperProcessor 会自动调用 tokenizer，把文字转成 token IDs，并放到：

# labels

In [18]:
#第三步：filtering（处理length）
max_input_length = 30.0


def is_audio_in_length_range(length):
    return length < max_input_length

In [19]:
#通过数据集的 .filter 方法，将我们的过滤功能应用到了训练数据集的所有样本上
corpus["train"] = corpus["train"].filter(is_audio_in_length_range,input_columns=["input_length"])

Filter:   0%|          | 0/2654 [00:00<?, ? examples/s]

In [20]:
# 在进入 Data Collator 之前，input_features 已经是 log-Mel spectrogram 了。

# 你前面这一步就完成了转换：

# example = processor(
#     audio=audio["array"],
#     sampling_rate=audio["sampling_rate"],
#     text=example["transcription"],
# )

# 这里的 processor 是 WhisperProcessor，内部其实同时做了两件事：

# audio waveform
#    ↓
# WhisperFeatureExtractor
#    ↓
# log-Mel spectrogram
#    ↓
# input_features

# transcription
#    ↓
# WhisperTokenizer
#    ↓
# token IDs
#    ↓
# labels

In [21]:
#data collator 把 DataLoader 取出来的多条单独样本，整理成一个可以直接送进 Whisper 训练的 batch。
# 它主要处理两件事：

# 处理音频 input_features
# Whisper 的音频已经被处理成固定 30 秒对应的 log-Mel。
# 所以这里基本不需要再补长度。
# 主要就是把多条数据 stack 成 PyTorch Tensor：
# sample 1: [80, 3000]
# sample 2: [80, 3000]
# sample 3: [80, 3000]
# sample 4: [80, 3000]

#         ↓ Data Collator

# batch input_features:
# [4, 80, 3000]

# 其中 4 就是 batch size。

# 处理文本 labels
# 文本长度不一样，比如：
# sample 1: [12, 35, 89, 3]
# sample 2: [18, 44, 7]
# sample 3: [25, 10, 11, 90, 6]

# 它们没办法直接组成矩阵，所以 tokenizer 会把它们 padding 到当前 batch 中最长的长度：

# [12, 35, 89,  3, PAD]
# [18, 44,  7, PAD, PAD]
# [25, 10, 11, 90,  6]

# 然后再把 PAD 替换成 -100：

# [12, 35, 89,   3, -100]
# [18, 44,  7, -100, -100]
# [25, 10, 11,  90,    6]

# 因为 PyTorch 的 Cross Entropy Loss 默认可以用：

# ignore_index=-100

# 于是这些 padding 位置就不会参与 loss 计算。

In [22]:
#  features 大概是一个 Python list：

# features = [
#     {
#         "input_features": ...,
#         "labels": ...
#     },
#     {
#         "input_features": ...,
#         "labels": ...
#     },
#     {
#         "input_features": ...,
#         "labels": ...
#     }
# ]

# 也就是：

# features
# ├── sample 1
# ├── sample 2
# └── sample 3

# 其中每一个 feature 就是一条样本：

# feature = {
#     "input_features": ...,
#     "labels": ...
# }

# 关键是这里：

# feature["input_features"]

# 前面你用 processor() 处理单条音频时，Whisper 的 feature extractor 通常返回的 input_features 会带一个最外层 batch 维度。

# 例如一条音频本身的 log-Mel 实际形状应该是：

# 80 × 3000

# 但是 processor 的输出可能是：

# [1, 80, 3000]

# 这里：

# 1     = 当前只有一条音频
# 80    = Mel bins
# 3000  = 时间帧

# 所以：

# feature["input_features"]

# 可以理解成：

# shape = [1, 80, 3000]

# 但 Data Collator 接下来希望拿到的是每条样本本身的：

# [80, 3000]

# 所以用了：

# feature["input_features"][0]

# 这里的 [0] 是：

# 取第 0 个样本，顺便去掉最外面的那个 1 维度。

# 例如：

# x.shape
# # (1, 80, 3000)

# x[0].shape
# # (80, 3000)

# 这不是在取“第一个 Mel bin”，而是在取最外层的第一条音频。

# 然后外面的：

# {"input_features": ...}

# 是重新包装成 feature extractor 的 .pad() 所要求的格式。

# 所以：

# {"input_features": feature["input_features"][0]}

# 最后变成：

# {
#     "input_features": 一个 [80, 3000] 的 log-Mel
# }

# 如果 batch 里有 3 条数据，那么这个 list comprehension：

# input_features = [
#     {"input_features": feature["input_features"][0]}
#     for feature in features
# ]

# 最后得到：

# input_features = [
#     {"input_features": logmel_1},   # [80, 3000]
#     {"input_features": logmel_2},   # [80, 3000]
#     {"input_features": logmel_3},   # [80, 3000]
# ]


In [23]:
# 一批 samples
#    │
#    ├── input_features（已经是 log-Mel）
#    │      ↓
#    │   组成 batch tensor
#    │
#    └── labels（token IDs，长度不同）
#           ↓
#        padding
#           ↓
#        PAD → -100
#           ↓
#        删除重复 BOS
#           ↓
# 最终 batch → Whisper → loss

In [24]:
#第四步：定义data collator
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        # features 是一个 batch 中的多条样本。
        # 每条样本已经经过前面的 processor 预处理：
        #   input_features -> log-Mel spectrogram
        #   labels         -> transcription 对应的 token IDs

        # ① 单独取出每条样本的 input_features（log-Mel）
        # 因为 Whisper 的 input_features 已经 padding 到固定 30s，
        # 所以这里主要是把多条样本组合成 PyTorch batch tensor，
        # 而不是再次进行真正的 padding。
        input_features = [
            {"input_features": feature["input_features"][0]}
            for feature in features
        ]

        # 使用 feature_extractor.pad 将多个 input_features
        # 整理成一个 batch，并转换为 PyTorch tensor
        # 例如 batch_size=4:
        # [80, 3000] × 4 -> [4, 80, 3000]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # ② 单独取出 labels（文本对应的 token IDs）
        # 不同 transcription 长度不同，所以 labels 长度也不同
        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        #把当前 batch 里长度不同的文本 token 序列，补齐到相同长度，并转成 PyTorch Tensor。

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        # ③ 将 label 中 padding 的位置替换为 -100
        #
        # attention_mask:
        # 真实 token -> 1
        # padding   -> 0
        #
        # masked_fill(..., -100):
        # 将 attention_mask != 1 的位置设为 -100
        #
        # CrossEntropyLoss 会忽略 target=-100 的位置，
        # 因此 padding token 不参与 loss 计算。
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        # ④ 如果所有 label 的第一个 token 都是 BOS token，
        # 就把这个 BOS token 删除。
        #
        # 原因：Whisper 在训练/decoder 输入时会自动添加 decoder start token，
        # 如果这里不删除，就可能重复添加。
        if (
            labels[:, 0] == self.processor.tokenizer.bos_token_id
        ).all().cpu().item():
            labels = labels[:, 1:]

        # ⑤ 将处理好的 labels 加入 batch
        # 最终 batch 大致为：
        # {
        #     "input_features": [batch_size, 80, 3000],
        #     "labels":         [batch_size, max_label_length]
        # }
        batch["labels"] = labels

        # 返回给 Trainer / Model 进行训练
        return batch

In [ ]:
# 假设：
# features = [
#     {"input_features": [[[1,2,3], [4,5,6]]]},
#     {"input_features": [[[7,8,9], [10,11,12]]]},
#     {"input_features": [[[13,14,15], [16,17,18]]]},
#     {"input_features": [[[19,20,21], [22,23,24]]]},
# ]

# 为了方便理解，我这里假设每条语音的 log-Mel 只有：

# 2个 Mel bins × 3个时间帧

# 4 条样本经过 list comprehension 后：

# input_features = [
#     {
#         "input_features": [
#             [1,2,3],
#             [4,5,6]
#         ]
#     },

#     {
#         "input_features": [
#             [7,8,9],
#             [10,11,12]
#         ]
#     },

#     {
#         "input_features": [
#             [13,14,15],
#             [16,17,18]
#         ]
#     },

#     {
#         "input_features": [
#             [19,20,21],
#             [22,23,24]
#         ]
#     }
# ]

In [25]:
#initialize data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [26]:
!pip install evaluate jiwer
import evaluate

metric = evaluate.load("cer")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.6 MB/s eta 0:00:00


In [27]:
#第五步：定义metrics
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# 文本标准化工具：
# 用于去除大小写、标点等差异，使 WER 更关注文本内容本身
normalizer = BasicTextNormalizer()


def compute_metrics(pred):
    # 模型预测得到的 token IDs
    pred_ids = pred.predictions

    # ground truth 的 label token IDs
    label_ids = pred.label_ids

    # 训练时为了忽略 padding，label 中的 padding 被设成了 -100。
    # 但 batch_decode 无法正常解码 -100，
    # 所以先将 -100 恢复成 tokenizer 的 pad_token_id。
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # 将预测 token IDs 解码成文本
    # skip_special_tokens=True：
    # 去掉 BOS、EOS、PAD 等特殊 token
    pred_str = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    # 将真实 label token IDs 解码成文本
    label_str = processor.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    # ① 计算原始文本的 CER（Orthographic CER）
    # 不进行文本标准化，因此大小写、标点、拼写形式等差异都会影响 CER
    cer_ortho = 100 * metric.compute(
        predictions=pred_str,
        references=label_str
    )

    # ② 对预测文本和真实文本进行标准化
    # 例如去除大小写、部分标点等差异
    pred_str_norm = [
        normalizer(pred)
        for pred in pred_str
    ]

    label_str_norm = [
        normalizer(label)
        for label in label_str
    ]

    # ③ 过滤掉标准化之后 reference 变成空字符串的样本
    # 因为空 reference 无法正常用于 CER 计算
    #
    # 注意：prediction 和 reference 必须同步过滤，
    # 否则两边样本就会错位。
    pred_str_norm = [
        pred_str_norm[i]
        for i in range(len(pred_str_norm))
        if len(label_str_norm[i]) > 0
    ]

    label_str_norm = [
        label_str_norm[i]
        for i in range(len(label_str_norm))
        if len(label_str_norm[i]) > 0
    ]

    # ④ 使用标准化后的文本计算 CER
    # 这个 CER 更关注“识别内容是否正确”，
    # 不太受大小写和标点格式的影响。
    cer = 100 * metric.compute(
        predictions=pred_str_norm,
        references=label_str_norm
    )

    # 返回两个指标：
    # cer_ortho：原始文本 WER
    # cer：标准化后的 WER
    return {
        "cer_ortho": cer_ortho,
        "cer": cer
    }

In [28]:
#load the pre-trained Whisper small checkpoint
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

In [29]:
#                Whisper
#                   │
#         ┌─────────┴─────────┐
#         ↓                   ↓
#       训练                 推理/生成
#         │                   │
# use_cache=False                use_cache=True
#         │                   │
# 兼容 gradient                加速 decoder
# checkpointing                 自回归生成
#                             │
#                    language=Sinhalese
#                    task=transcribe

In [30]:
from functools import partial

# 训练阶段关闭 KV cache：
# gradient checkpointing 会通过重新计算中间激活来节省显存，
# 与 use_cache=True 不兼容，因此训练时关闭 cache。
model.config.use_cache = False

# 之后调用 model.generate(...) 时，
# 相当于自动带上这三个参数。
model.generate = partial(
    model.generate,
    language="chinese",
    task="transcribe",
    use_cache=True
)

In [31]:
# 第六步：定义 training configuration
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(

    # 训练结果保存目录
    output_dir="./whisper-small-dv",

    # 每张 GPU 每次训练的 batch size
    per_device_train_batch_size=16,

    # 梯度累积
    gradient_accumulation_steps=1,

    # 学习率
    learning_rate=1e-5,

    # warmup 后保持固定学习率
    lr_scheduler_type="constant_with_warmup",

    # warmup 50 steps
    warmup_steps=50,

    # 总训练 500 个 optimizer steps
    max_steps=500,

    # 降低显存占用
    gradient_checkpointing=True,

    # FP16 训练
    fp16=True,

    # FP16 evaluation
    fp16_full_eval=True,

    # =========================
    # Training Progress
    # =========================

    # 按 step 记录训练日志
    logging_strategy="steps",

    # 每 10 steps 输出一次 training loss / learning rate
    logging_steps=10,

    # 第一个 step 就输出日志
    logging_first_step=True,

    # 显示 tqdm 训练进度条
    disable_tqdm=False,

    # =========================
    # Evaluation
    # =========================

    # 按 step 进行 validation
    eval_strategy="steps",

    # 每 100 steps evaluation 一次
    # 会计算 Validation Loss + CER
    eval_steps=100,

    per_device_eval_batch_size=16,

    # evaluation 时使用 generate() 生成文本
    predict_with_generate=True,

    # 最大生成长度
    generation_max_length=225,

    # =========================
    # Checkpoint
    # =========================

    # 按 step 保存 checkpoint
    save_strategy="steps",

    # 每 100 steps 保存一次
    save_steps=100,

    # TensorBoard
    report_to=["tensorboard"],

    # 训练结束后加载 CER 最低的 checkpoint
    load_best_model_at_end=True,

    # 根据 CER 判断 best model
    metric_for_best_model="cer",

    # CER 越低越好
    greater_is_better=False,

    # 上传 HF Hub
    push_to_hub=True,
)

In [34]:
#定义trainer
from transformers import Seq2SeqTrainer

# 将 push_to_hub 设为 False 避免 401 认证错误
training_args.push_to_hub = False

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=corpus["train"],
    eval_dataset=corpus["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [35]:
#开始训练
trainer.train()

Step,Training Loss,Validation Loss,Cer Ortho,Cer
100,0.999008,0.978870,53.647174,53.223274
200,0.542573,0.738946,41.081729,40.500650
300,0.484008,0.643626,38.628095,38.024315


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Cer Ortho,Cer
100,0.999008,0.978870,53.647174,53.223274
200,0.542573,0.738946,41.081729,40.500650
300,0.484008,0.643626,38.628095,38.024315
400,0.256199,0.628479,34.821191,34.224575
500,0.248738,0.621131,33.388056,32.748643


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=500, training_loss=0.8178065805435181, metrics={'train_runtime': 4486.905, 'train_samples_per_second': 1.783, 'train_steps_per_second': 0.111, 'total_flos': 2.30695169015808e+18, 'train_loss': 0.8178065805435181, 'epoch': 3.0120481927710845})

In [36]:
#把key-word arguments (kwargs)保存，上传到huggingface
kwargs = {
    "dataset_tags": "TingChen-ppmc/Shanghai_Dialect_Conversational_Speech_Corpus",
    "dataset": "Shanghai Dialect Conversational Speech Corpus",
    "language": "zh",
    "model_name": "Whisper Small Fine-tuned on 不懂上海话的老王",
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}

In [38]:
#登录我的hugging账号
from huggingface_hub import notebook_login

notebook_login()

In [39]:
#上传checkpoints配置
trainer.push_to_hub(**kwargs)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/wangyuzhe/whisper-small-dv/commit/24499300b9c3e664f9e257c94531996b18befb01', commit_message='End of training', commit_description='', oid='24499300b9c3e664f9e257c94531996b18befb01', pr_url=None, repo_url=RepoUrl('https://huggingface.co/wangyuzhe/whisper-small-dv', endpoint='https://huggingface.co', repo_type='model', repo_id='wangyuzhe/whisper-small-dv'), pr_revision=None, pr_num=None)